# Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pulp as pl
from IPython.display import clear_output
import yfinance as yf
from scipy.stats import norm
from pulp import *

# Input Data

In [2]:
budget = 100000
risk = 3
expected_return = 80
stocks = ["AAPL","^GSPC"]
days = 365
diver = 1

# VaR 

In [3]:
def VaR(ticker, p=0.95):
    
    # fetch all historical data for the ticker
    data = yf.download(ticker, progress=False,auto_adjust=False)
    if data.empty:
        raise ValueError(f"No historical data found for ticker: {ticker}")

    returns = (data['Open']-data['Close'])/data['Open'] * -1
    return returns[returns.columns[0]].quantile(p)

# RoI

In [4]:
def RoI(ticker, p=0.95,days=365):
    import yfinance as yf
    from scipy.stats import norm

    # fetch all historical data for the ticker
    data = yf.download(ticker,start="1900-01-01" ,progress=False,auto_adjust=False)
    if data.empty:
        raise ValueError(f"No historical data found for ticker: {ticker}")

    returns = (data['Open']-data['Close'].shift(days))/data['Open']
    returns = returns[returns.columns[0]]
    E = returns.mean()
    return E 

# Optimization model

### pulp 2

In [5]:
def optimize2(tickers,budget,VaRs,RoIs,mxr,exr,ponder=0.5,diver=0.5):
    
    if mxr >= 1:
        mxr = mxr / 100
    if exr >= 1:
        exr = exr / 100

    # Penalty weights
    M1 = budget * ponder  # risk penalty
    M2 = budget * (1 - ponder)  # return penalty

    # Create the model
    model = LpProblem("Portfolio_Optimization", LpMaximize)

    # Decision variables
    stocks = {t: LpVariable(f"stocks_{t}", lowBound=0) for t in tickers}
    s1 = LpVariable("s1")  # risk slack
    s2 = LpVariable("s2")  # return slack
    ts = LpVariable("ts", lowBound=0)  # total spent
    d = {t: LpVariable(f"d_{t}", lowBound=0, upBound=1,cat=LpBinary) for t in tickers}  # diversification binaries

    # Objective function: maximize return minus penalties
    model += (
        lpSum([stocks[t] * RoIs[t] for t in tickers]) - s1 - s2,
        "Total_Return_Minus_Penalties"
    )


    # Constraints
    # Budget constraint
    model += lpSum([stocks[t] for t in tickers]) <= budget, "Max_Expenditure"

    # Total spent continuity
    model += lpSum([stocks[t] for t in tickers]) == ts, "Total_Spent"

    # Risk constraint (with slack)
    model += lpSum([stocks[t] * VaRs[t] for t in tickers]) == ts * mxr + s1, "Risk_Constraint"

    # Return constraint (with slack)
    model += lpSum([stocks[t] * RoIs[t] for t in tickers]) == ts * exr - s2, "Return_Constraint"

    # Diversification constraint
    model += lpSum([d[t] for t in tickers]) <= diver * len(tickers), "Diversification_Constraint"
    for t in tickers:
        model += stocks[t] <= d[t] * budget, f"Diversification_Link_{t}"

    # Solve
    model.solve()
    clear_output()

    class result:
        def __init__(self):
            pass
         
    r = result(); r.model = model; r.stocks = stocks; r.s1 = s1; r.s2 = s2; r.ts = ts

    return r

# Report

In [6]:
def generate_report(budget, risk, expected_return, tickers, days):
    
    VaRs = {ticker: VaR(ticker) for ticker in stocks}
    RoIs = {ticker: RoI(ticker, days=days) for ticker in stocks}


    z = optimize2(tickers = stocks,
                budget=budget,
                VaRs=VaRs,
                RoIs=RoIs,
                mxr=risk,
                exr=expected_return,
                diver =diver
                )

    varvals = {v.name: v.varValue for v in z.model.variables()}

    # print inputs
    print("Inputs:")
    print("#"*20)
    inputs =pd.DataFrame({
        'Budget':"$"+str(budget),
        "Stocks":", ".join(stocks),
        "Risk (VaR)":str(risk)+"%",
        "Expected Return": str(expected_return)+"%",
        "Days": days,
    },index=["Value"]).T
    print(inputs)

    # evironment
    print("\nEnvironment Parameters:\n"+"#"*20)
    envs =pd.DataFrame({'VaR':VaRs,'RoI':RoIs},index=stocks)
    print(envs)


    # variables
    print("\nVaribles:\n"+"#"*20)
    vars =pd.DataFrame([{v.name:v.varValue for v in z.model.variables()},
                        {v.name:v.dj for v in z.model.variables()}],
                        index=["Value","Reduced cost"]).T.map(lambda x: round(x,4))
    print(vars)


    # constraints
    print("\nConstraints:\n"+"#"*20)
    constraints = pd.DataFrame([{c.name:c.pi for c in z.model.constraints.values()},
                        {c.name:c.slack for c in z.model.constraints.values()},
                        {c.name:c.value() for c in z.model.constraints.values()}],
                        index=["Dual Value","Slack","Value"]).T
    print(constraints)


    print("\nEstadísticas de la solución:\n"+"#"*30)
    stats =(pd.DataFrame({
            "Risk": sum([varvals[f'stocks_{t}'] * VaRs[t] for t in stocks]),
            "Expected Return": sum([varvals[f'stocks_{t}'] * RoIs[t] for t in stocks]),
            "Pergcentage invested": z.ts.varValue**2/budget,
            "Number of stocks": sum([1 for t in stocks if varvals[f'stocks_{t}']> 0.0])*z.ts.varValue/100
        },index=["Value"]).T/z.ts.varValue*100).map(lambda x: round(x,2))
    print(stats)

    reccomendations = []
    # ponderation


    # diversification analysis
    if constraints[constraints.index.str.startswith("Diversification_Link")]["Dual Value"].sum() > 0:
        reccomendations.append("Consider increasing diversification to reduce risk.")
    elif constraints[constraints.index.str.startswith("Diversification_Link")]["Dual Value"].sum() == 0:
        reccomendations.append("Diversification level is adequate.")
    else:
        reccomendations.append("Consider decreasing diversification to increase returns.")

    return z, inputs, envs, vars, constraints, stats

In [7]:

VaRs = {ticker: VaR(ticker) for ticker in stocks}
RoIs = {ticker: RoI(ticker, days=days) for ticker in stocks}


z = optimize2(tickers = stocks,
              budget=budget,
              VaRs=VaRs,
              RoIs=RoIs,
              mxr=risk,
              exr=expected_return,
              diver =diver
            )

varvals = {v.name: v.varValue for v in z.model.variables()}

# print inputs
print("Inputs:")
print("#"*20)
inputs =pd.DataFrame({
    'Budget':"$"+str(budget),
    "Stocks":", ".join(stocks),
    "Risk (VaR)":str(risk)+"%",
    "Expected Return": str(expected_return)+"%",
    "Days": days,
},index=["Value"]).T
print(inputs)

# evironment
print("\nEnvironment Parameters:\n"+"#"*20)
envs =pd.DataFrame({'VaR':VaRs,'RoI':RoIs},index=stocks)
print(envs)


# variables
print("\nVaribles:\n"+"#"*20)
vars =pd.DataFrame([{v.name:v.varValue for v in z.model.variables()},
                    {v.name:v.dj for v in z.model.variables()}],
                    index=["Value","Reduced cost"]).T.map(lambda x: round(x,4))
print(vars)


# constraints
print("\nConstraints:\n"+"#"*20)
constraints = pd.DataFrame([{c.name:c.pi for c in z.model.constraints.values()},
                    {c.name:c.slack for c in z.model.constraints.values()},
                    {c.name:c.value() for c in z.model.constraints.values()}],
                    index=["Dual Value","Slack","Value"]).T
print(constraints)


print("\nEstadísticas de la solución:\n"+"#"*30)
stats =(pd.DataFrame({
        "Risk": sum([varvals[f'stocks_{t}'] * VaRs[t] for t in stocks]),
        "Expected Return": sum([varvals[f'stocks_{t}'] * RoIs[t] for t in stocks]),
        "Pergcentage invested": z.ts.varValue**2/budget,
        "Number of stocks": sum([1 for t in stocks if varvals[f'stocks_{t}']> 0.0])*z.ts.varValue/100
    },index=["Value"]).T/z.ts.varValue*100).map(lambda x: round(x,2))
print(stats)

reccomendations = []
# ponderation


# diversification analysis
if constraints[constraints.index.str.startswith("Diversification_Link")]["Dual Value"].sum() > 0:
    reccomendations.append("Consider increasing diversification to reduce risk.")
elif constraints[constraints.index.str.startswith("Diversification_Link")]["Dual Value"].sum() == 0:
    reccomendations.append("Diversification level is adequate.")
else:
    reccomendations.append("Consider decreasing diversification to increase returns.")


Inputs:
####################
                       Value
Budget               $100000
Stocks           AAPL, ^GSPC
Risk (VaR)                3%
Expected Return          80%
Days                     365

Environment Parameters:
####################
            VaR       RoI
AAPL   0.019630  0.104027
^GSPC  0.007176  0.050575

Varibles:
####################
              Value  Reduced cost
d_AAPL          0.0        0.0000
d_^GSPC         0.0        0.0000
s1              0.0        0.0000
s2              0.0        0.0000
stocks_AAPL     0.0       -0.5816
stocks_^GSPC    0.0       -0.6760
ts              0.0        0.0000

Constraints:
####################
                            Dual Value     Slack     Value
Max_Expenditure                  -0.00  100000.0 -100000.0
Total_Spent                       0.77      -0.0       0.0
Risk_Constraint                   1.00      -0.0       0.0
Return_Constraint                -1.00      -0.0       0.0
Diversification_Constraint       -0.00 

In [8]:
if stats.loc["Number of stocks","Value"] < len(stocks):
    if vars.loc["s1","Value"] < 0:
        print("You can increase the risk to potentially improve returns.")
    if vars.loc["s2","Value"] < 0:
        print("You can decrease the expected return to potentially reduce risk.")

In [12]:
import yfinance as yf
yf.download("AAPL",period="max",progress=False,auto_adjust=False)["Close"]["AAPL"].shape

(11329,)